
# C6-pytorch — Practice p13

**Type:** integrative (parts consume earlier results) · **Difficulty:** core · **Concepts:** nn-module, manual-weights, threshold-activation

One network, two implementations, zero disagreement.

The hand-set weights below define a $2 \to 3 \to 1$ pipeline:
dense layer, threshold gate, dense layer — **no** final gate, so the
output is a real number, not a 0/1 verdict.

**(a) NumPy first.** Using the supplied C5 helpers, push `X_np`
through the pipeline: `H_np = step_activation(affine_layer(X_np, W1, b1))`,
then `out_np = affine_layer(H_np, W2, b2)` — shape `(5, 1)`.

**(b) Torch modules.** Implement `TinyNet(nn.Module)` whose
constructor takes the four weight tensors and stores **two
`DenseLayer` submodules and one `ThresholdGate`** as attributes, and
whose `forward` computes the same pipeline.
Build `net` from `torch.as_tensor(...)` copies of the same weights,
and compute `out_t = net(X_t)` where
`X_t = torch.from_numpy(X_np).to(torch.float64)` (the explicit cast is
part of the contract).

**(c) The reconciliation.** `gap = float(np.abs(out_t.numpy() - out_np).max())`
— it must be exactly `0.0`. Also collect
`param_names = sorted(name for name, _ in net.named_parameters())` and
`n_tensors = len(param_names)`.

**(d)** In the markdown cell: one or two sentences — *which* mechanism
put those four names into `named_parameters()`, and why the gate
appears in none of them.

**Banned (zero points): `nn.Linear`; `nn.Sequential`; recomputing
`out_t` with NumPy instead of the module.**


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (Session 1)

import numpy as np


def affine_layer(x, W, b):
    """C5's dense-layer map: rows of W are output units; returns x @ W.T + b."""
    return x @ W.T + b


def step_activation(z):
    """C5's threshold activation: 1.0 where z >= 0, else 0.0."""
    return (z >= 0).astype(float)


class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


W1 = np.array([[1.0, 0.0],
               [0.0, 1.0],
               [1.0, 1.0]])
b1 = np.array([0.0, 0.0, -1.0])
W2 = np.array([[1.5, -1.0, 2.0]])
b2 = np.array([0.25])

X_np = np.array([[0.0, 0.0],
                 [1.0, 0.0],
                 [0.0, 1.0],
                 [2.0, 2.0],
                 [-1.0, 3.0]])

# (a) YOUR CODE HERE
H_np = ...
out_np = ...

# (b) YOUR CODE HERE — class TinyNet, then:
net = ...
X_t = ...
out_t = ...

# (c) YOUR CODE HERE
gap = ...
param_names = ...
n_tensors = ...


*Your (d) answer here (1–2 sentences).*